# AI Cycling Coach — GPU Training (Kaggle)

**Settings (right sidebar) → Accelerator → GPU T4 x2** before running.

Then click **Run All**. No uploads needed — generates data here (~3 min) then trains on GPU (~1–2 h).

In [ ]:
# ── 1. Check GPU ─────────────────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('No GPU — go to Settings → Accelerator → GPU T4 x2')

In [ ]:
# ── 2. Clone repo + set paths ────────────────────────────────────────────────
import os, sys

REPO    = 'https://github.com/yossibello/ai-coach.git'
WORKDIR = '/kaggle/working/ai-coach'

if not os.path.exists(WORKDIR):
    !git clone {REPO} {WORKDIR}
else:
    !cd {WORKDIR} && git pull

%cd {WORKDIR}

for p in [f'{WORKDIR}/backend', WORKDIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

os.environ['PYTHONPATH']      = f'{WORKDIR}/backend'
os.environ['PYTHONIOENCODING'] = 'utf-8'

!mkdir -p ml/data backend/models
print('cwd:', os.getcwd())
print('sys.path[0:3]:', sys.path[:3])

In [ ]:
# ── 3. Install dependencies ──────────────────────────────────────────────────
!pip install pyarrow --upgrade -q
import torch, pandas
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
print('pandas:', pandas.__version__)

In [ ]:

# ── 4. Load training data ────────────────────────────────────────────────────
# TWO OPTIONS — set MODE below:
#
#   'dataset'  → you added the ai-coach-synthetic Kaggle dataset (recommended, instant)
#   'generate' → generate here in Kaggle (~3 min for 20K, ~8 min for 50K)

import os, multiprocessing, pandas as pd

MODE          = 'dataset'   # ← 'dataset' | 'generate'
ATHLETES      = 20_000      # only used when MODE='generate'
DATASET_PATH  = '/kaggle/input/ai-coach-synthetic/synthetic.parquet'  # adjust if dataset name differs
DATA_FILE     = 'ml/data/synthetic.parquet'

os.makedirs('ml/data', exist_ok=True)

if MODE == 'dataset':
    if not os.path.exists(DATASET_PATH):
        raise FileNotFoundError(
            f'Dataset not found at {DATASET_PATH}\n'
            'Add it: right panel → Add data → search "ai-coach-synthetic"'
        )
    print(f'Using Kaggle dataset: {DATASET_PATH}  ({os.path.getsize(DATASET_PATH)/1e6:.0f} MB)')
    DATA_FILE = DATASET_PATH   # read directly — no copy needed

elif MODE == 'generate':
    if os.path.exists(DATA_FILE):
        print(f'✓ Already generated ({os.path.getsize(DATA_FILE)/1e6:.0f} MB), skipping.')
    else:
        workers = max(1, multiprocessing.cpu_count() - 1)
        print(f'Generating {ATHLETES:,} athletes using {workers} workers…')
        !python -m ml.training.generate_synthetic \
            --athletes {ATHLETES} \
            --workers  {workers} \
            --output   {DATA_FILE}

df = pd.read_parquet(DATA_FILE)
assert 'risk_ot_class'   in df.columns, 'Old parquet — regenerate with latest generate_synthetic.py'
assert 'risk_inj_target' in df.columns, 'Old parquet — regenerate with latest generate_synthetic.py'
print(f'✓ Data ready: {len(df):,} rows, {df.athlete_id.nunique():,} athletes, {len(df.columns)} cols')
del df


In [ ]:

# ── 5. Train ─────────────────────────────────────────────────────────────────
import sys, os, torch, argparse

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
if   vram_gb >= 45: BATCH_SIZE = 4096  # A6000 48 GB / A100 80 GB
elif vram_gb >= 38: BATCH_SIZE = 2048  # A100 40 GB / H100
elif vram_gb >= 20: BATCH_SIZE = 1024  # RTX 3090+
else:               BATCH_SIZE = 1024  # T4 16 GB — 8M-param model uses ~3 GB, plenty of headroom

# With 11M sequences the full epoch is ~10K batches at bs=1024.
# Cap at 3000 steps/epoch (3.07M samples — 28% of data per epoch, ~15 min on T4).
# Combined with bs=1024 this is 6× more data/epoch than the previous 2000×256 setting.
# Validation still runs on the full val split.
STEPS_PER_EPOCH = 3000   # set None to use all batches (much slower per epoch)

EPOCHS     = 50  # each epoch sees 6× more data; 50 epochs ≈ 3× total samples vs old 100
MODEL_FILE = 'backend/models/cycling_coach.pt'

print(f'GPU VRAM: {vram_gb:.1f} GB → batch size: {BATCH_SIZE}')
print(f'Steps/epoch cap: {STEPS_PER_EPOCH}  (set STEPS_PER_EPOCH=None for full epoch)')
print(f'Data: {DATA_FILE}  ({os.path.getsize(DATA_FILE)/1e6:.0f} MB)')
print('─' * 60)

from ml.training.train import train as run_training

os.makedirs(os.path.dirname(MODEL_FILE), exist_ok=True)

args = argparse.Namespace(
    data             = DATA_FILE,
    output           = MODEL_FILE,
    checkpoint       = None,
    epochs           = EPOCHS,
    batch_size       = BATCH_SIZE,
    steps_per_epoch  = STEPS_PER_EPOCH,
    lr               = 3e-4,
    seq_len          = 90,
    val_frac         = 0.1,
    seed             = 42,
    d_model          = 256,   # was 128 — 2× wider embeddings
    nhead            = 8,
    num_layers       = 8,     # was 6 — 2 more transformer layers
    d_ff             = 1024,  # was 512 — proportional to d_model
    dropout          = 0.1,
    fast             = False,
    patience         = 20,
    compile          = False,
    no_amp           = False,
)

run_training(args)
print(f'\n✓ Training complete! Model → {MODEL_FILE}')


In [ ]:
# ── 6. Copy model to /kaggle/working/ so Kaggle saves it as output ────────────
import shutil, os

OUT = '/kaggle/working/cycling_coach.pt'
shutil.copy(MODEL_FILE, OUT)
print(f'✓ Model saved to {OUT}')
print('  → After the notebook finishes, go to the Output tab and download it.')

In [ ]:
# ── 7. (Optional) Push model to GitHub ──────────────────────────────────────
# Create a PAT at https://github.com/settings/tokens (Classic, repo scope)
# then paste it when prompted.

from getpass import getpass
token = getpass('GitHub PAT (hidden): ')

!git config user.email 'kaggle@training'
!git config user.name  'Kaggle Training'
!git remote set-url origin https://{token}@github.com/yossibello/ai-coach.git
!cp /kaggle/working/cycling_coach.pt backend/models/cycling_coach.pt
!git add backend/models/cycling_coach.pt
!git commit -m "Trained model: {ATHLETES} athletes, {EPOCHS} epochs (Kaggle GPU)"
!git push origin main
print('✓ Model pushed to GitHub!')

In [ ]:
# ── 8. Sanity check ──────────────────────────────────────────────────────────
import torch, sys
from app.ml.model import CyclingTransformer

ckpt = torch.load(MODEL_FILE, map_location='cpu')
cfg  = ckpt.get('config', {})
m    = CyclingTransformer(
    d_model=cfg.get('d_model', 128),
    nhead=cfg.get('nhead', 8),
    num_layers=cfg.get('num_layers', 6),
    dim_feedforward=cfg.get('dim_feedforward', 512),
)
m.load_state_dict(ckpt['state_dict'])
m.eval()
print('Model loaded OK')
print('Params:', sum(p.numel() for p in m.parameters()))
print('Best val loss:', ckpt.get('metrics', {}).get('val_loss', 'n/a'))
print('Epoch:',        ckpt.get('metrics', {}).get('epoch',    'n/a'))